**Feature Selection Method:** PSO   
**Dataset:** QUT-DV25 (Opensnoop)   
**Developed By:** eDySec Research Team   
**Plartform:** Ubuntu

All experiments in this notebook were conducted using **Python 3.10** with the following libraries:

`pandas==1.5.3`,  
`scikit-learn==1.2.2`,  
`openpyxl`,  
`numpy==1.23.5`,  
`scipy==1.9.3`,  
`tensorflow==2.11.0`,  
`matplotlib==3.7.1`,  
`seaborn==0.12.2`,  
`joblib==1.3.2`,  
`shap==0.41.0`,  
`lime`,  
`flaml[automl]==2.5.0`,  
`notebook==6.5.6`,  
`pywinpty==2.0.10`  (Only for windows)  `threadpoolctl==3.1.0` (for Ubuntu)   
`terminado==0.17.1`,  
`transformers==4.49.0`.

#### Full Environment Setup: https://github.com/tanzirmehedi/eDySec

These versions were used to ensure **consistent and reproducible experimental results**.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack
from scipy.sparse import csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support, cohen_kappa_score
)
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import gc
from sklearn.feature_selection import f_classif
from flaml import AutoML

In [ ]:

# This will allow TensorFlow to allocate as much GPU memory as needed, including full 16GB if needed.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Disable memory growth (optional - default is False)
        tf.config.experimental.set_memory_growth(gpus[0], False)
        print("Memory growth disabled (default behavior).")
    except RuntimeError as e:
        print("Error:", e)

Memory growth disabled (default behavior).


In [ ]:
DATASET_NAME = "Opensnoop"
file_path = 'QUT-DV25_'+DATASET_NAME+'_Traces.csv'
data = pd.read_csv(file_path)

In [ ]:
gc.collect()
tf.keras.backend.clear_session()


In [ ]:
data.head()

,Package_Name,Total_Paths,Total_Error,Total_ File_Descriptor,Python_Related_Keywords,Install_Package_Keywords,Root_DIR_Installation,Temporary_DIR_Installation,Home_DIR_Installation,User_Access,Sys_Access,Etc_DIR_Installation,Other_DIR_Installation,Level
0,10Cent10-999.0.4.tar.gz,8045,8045,29,1083,2305,0,71,792,411,1225,151,5395,1
1,10Cent11-999.0.4.tar.gz,4790,4790,12,1616,843,2,87,1118,721,1200,104,1558,1
2,11Cent-999.0.0.tar.gz,26159,26159,27,3299,4536,0,200,2485,1072,1236,450,20716,1
3,11Cent-999.0.1.tar.gz,11194,11194,28,1521,2558,2,97,1035,891,1211,265,7693,1
4,11Cent-999.0.2.tar.gz,13561,13561,35,2656,5265,501,17,1497,1719,1219,291,8317,1


In [ ]:
X_df = data.drop(columns=['Level']).copy()
y = data['Level']

# Encode label if needed
if y.dtype == 'object' or y.dtype.name == 'category':
    y = LabelEncoder().fit_transform(y)

# Identify categorical columns (strings)
cat_cols = X_df.select_dtypes(include='object').columns
num_cols = X_df.select_dtypes(include=['int64','float64']).columns

# Simple Label Encoding for categorical columns
for col in cat_cols:
    X_df[col] = LabelEncoder().fit_transform(X_df[col].astype(str))

# Now all columns are numeric
X = X_df.values
feature_names = list(X_df.columns)

# =====================
# Fitness Function
# =====================
def fitness_function(selected_features, X_train, y_train, X_val, y_val):
    if np.sum(selected_features) == 0:
        return 0
    selected_cols = np.where(selected_features == 1)[0]
    model = LogisticRegression(max_iter=500)
    model.fit(X_train[:, selected_cols], y_train)
    preds = model.predict(X_val[:, selected_cols])
    return accuracy_score(y_val, preds)

# =====================
# PSO Feature Selection (20 iterations)
# =====================
def pso_feature_selection(X, y, num_particles=5, max_iter=20):
    n_features = X.shape[1]
    particles = np.random.randint(2, size=(num_particles, n_features))
    velocities = np.random.uniform(-1, 1, (num_particles, n_features))

    pbest_positions = particles.copy()
    pbest_scores = np.zeros(num_particles)
    gbest_position = None
    gbest_score = -1

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Initialize pbest and gbest
    for i in range(num_particles):
        score = fitness_function(particles[i], X_train, y_train, X_val, y_val)
        pbest_scores[i] = score
        if score > gbest_score:
            gbest_score = score
            gbest_position = particles[i].copy()

    w, c1, c2 = 0.7, 1.5, 1.5
    for t in range(max_iter):
        for i in range(num_particles):
            r1, r2 = np.random.rand(), np.random.rand()
            velocities[i] = (
                w * velocities[i]
                + c1 * r1 * (pbest_positions[i] - particles[i])
                + c2 * r2 * (gbest_position - particles[i])
            )
            sigmoid = 1 / (1 + np.exp(-velocities[i]))
            particles[i] = (np.random.rand(n_features) < sigmoid).astype(int)

            score = fitness_function(particles[i], X_train, y_train, X_val, y_val)
            if score > pbest_scores[i]:
                pbest_scores[i] = score
                pbest_positions[i] = particles[i].copy()
                if score > gbest_score:
                    gbest_score = score
                    gbest_position = particles[i].copy()

        selected = [feature_names[i] for i in range(n_features) if gbest_position[i]==1]
        dropped = [feature_names[i] for i in range(n_features) if gbest_position[i]==0]
        print(f"Iteration {t+1}/{max_iter}, Best Accuracy: {gbest_score:.4f}")
        print(f"Selected features: {selected}")
        print(f"Dropped features: {dropped}\n")

    # Final features after all iterations
    final_selected = [feature_names[i] for i in range(n_features) if gbest_position[i]==1]
    final_dropped = [feature_names[i] for i in range(n_features) if gbest_position[i]==0]
    print("===== FINAL RESULTS AFTER 20 ITERATIONS =====")
    print(f"Best Accuracy: {gbest_score:.4f}")
    print(f"Final Selected features: {final_selected}")
    print(f"Final Dropped features: {final_dropped}")
    print(len(final_selected))
    print(len(final_dropped))

    return gbest_position, gbest_score

best_features, best_score = pso_feature_selection(X, y, num_particles=5, max_iter=20)


C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,


Iteration 1/20, Best Accuracy: 0.7958
Selected features: ['Package_Name', 'Total_Paths', 'Total_ File_Descriptor', 'Python_Related_Keywords', 'Root_DIR_Installation', 'Temporary_DIR_Installation', 'Home_DIR_Installation', 'User_Access', 'Sys_Access', 'Other_DIR_Installation']
Dropped features: ['Total_Error', 'Install_Package_Keywords', 'Etc_DIR_Installation']

Iteration 2/20, Best Accuracy: 0.7958
Selected features: ['Package_Name', 'Total_Paths', 'Total_ File_Descriptor', 'Python_Related_Keywords', 'Root_DIR_Installation', 'Temporary_DIR_Installation', 'Home_DIR_Installation', 'User_Access', 'Sys_Access', 'Other_DIR_Installation']
Dropped features: ['Total_Error', 'Install_Package_Keywords', 'Etc_DIR_Installation']

Iteration 3/20, Best Accuracy: 0.7958
Selected features: ['Package_Name', 'Total_Paths', 'Total_ File_Descriptor', 'Python_Related_Keywords', 'Root_DIR_Installation', 'Temporary_DIR_Installation', 'Home_DIR_Installation', 'User_Access', 'Sys_Access', 'Other_DIR_Installati

C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,


Iteration 5/20, Best Accuracy: 0.7961
Selected features: ['Package_Name', 'Total_Paths', 'Total_Error', 'Total_ File_Descriptor', 'Python_Related_Keywords', 'Root_DIR_Installation', 'Temporary_DIR_Installation', 'Home_DIR_Installation', 'User_Access', 'Sys_Access', 'Other_DIR_Installation']
Dropped features: ['Install_Package_Keywords', 'Etc_DIR_Installation']

Iteration 6/20, Best Accuracy: 0.7961
Selected features: ['Package_Name', 'Total_Paths', 'Total_Error', 'Total_ File_Descriptor', 'Python_Related_Keywords', 'Root_DIR_Installation', 'Temporary_DIR_Installation', 'Home_DIR_Installation', 'User_Access', 'Sys_Access', 'Other_DIR_Installation']
Dropped features: ['Install_Package_Keywords', 'Etc_DIR_Installation']



C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,


Iteration 7/20, Best Accuracy: 0.8025
Selected features: ['Total_Paths', 'Total_ File_Descriptor', 'Python_Related_Keywords', 'Install_Package_Keywords', 'Root_DIR_Installation', 'Temporary_DIR_Installation', 'Home_DIR_Installation', 'User_Access', 'Sys_Access']
Dropped features: ['Package_Name', 'Total_Error', 'Etc_DIR_Installation', 'Other_DIR_Installation']

Iteration 8/20, Best Accuracy: 0.8025
Selected features: ['Total_Paths', 'Total_ File_Descriptor', 'Python_Related_Keywords', 'Install_Package_Keywords', 'Root_DIR_Installation', 'Temporary_DIR_Installation', 'Home_DIR_Installation', 'User_Access', 'Sys_Access']
Dropped features: ['Package_Name', 'Total_Error', 'Etc_DIR_Installation', 'Other_DIR_Installation']

Iteration 9/20, Best Accuracy: 0.8025
Selected features: ['Total_Paths', 'Total_ File_Descriptor', 'Python_Related_Keywords', 'Install_Package_Keywords', 'Root_DIR_Installation', 'Temporary_DIR_Installation', 'Home_DIR_Installation', 'User_Access', 'Sys_Access']
Dropped f

C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,


Iteration 12/20, Best Accuracy: 0.8025
Selected features: ['Total_Paths', 'Total_ File_Descriptor', 'Python_Related_Keywords', 'Install_Package_Keywords', 'Root_DIR_Installation', 'Temporary_DIR_Installation', 'Home_DIR_Installation', 'User_Access', 'Sys_Access']
Dropped features: ['Package_Name', 'Total_Error', 'Etc_DIR_Installation', 'Other_DIR_Installation']

Iteration 13/20, Best Accuracy: 0.8025
Selected features: ['Total_Paths', 'Total_ File_Descriptor', 'Python_Related_Keywords', 'Install_Package_Keywords', 'Root_DIR_Installation', 'Temporary_DIR_Installation', 'Home_DIR_Installation', 'User_Access', 'Sys_Access']
Dropped features: ['Package_Name', 'Total_Error', 'Etc_DIR_Installation', 'Other_DIR_Installation']

Iteration 14/20, Best Accuracy: 0.8025
Selected features: ['Total_Paths', 'Total_ File_Descriptor', 'Python_Related_Keywords', 'Install_Package_Keywords', 'Root_DIR_Installation', 'Temporary_DIR_Installation', 'Home_DIR_Installation', 'User_Access', 'Sys_Access']
Droppe

In [ ]:
selected_features = ['Total_Paths', 'Total_ File_Descriptor', 'Python_Related_Keywords', 'Install_Package_Keywords', 'Root_DIR_Installation', 'Temporary_DIR_Installation', 'Home_DIR_Installation', 'User_Access', 'Sys_Access']
